In [79]:
import pandas as pd 
import numpy as np
import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

In [80]:
df = pd.read_csv("/kaggle/input/datasets/waseemalastal/imdb-dataset/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [81]:
import re 
import string 
exclude = string.punctuation

def pre_process_data(text):
    
    #Lower text 
    text=text.lower().strip()
    
    #Remove Html tags
    pattern = re.compile('<.*?>')
    text = pattern.sub(r'',text)
    
    #Remove puctuation 
    text = text.translate(str.maketrans('','',exclude))

    #Sent tokenized document 
    return text.split(" ")

df["review"] = df["review"].apply(pre_process_data)

In [82]:
df["sentiment"] = df["sentiment"].apply(lambda x:0 if x == 'negative' else 1)
df.head()

,review,sentiment
0,"[one, of, the, other, reviewers, has, mentione...",1
1,"[a, wonderful, little, production, the, filmin...",1
2,"[i, thought, this, was, a, wonderful, way, to,...",1
3,"[basically, theres, a, family, where, a, littl...",0
4,"[petter, matteis, love, in, the, time, of, mon...",1


In [83]:
df["sentiment"].value_counts()

sentiment
1    25000
0    25000
Name: count, dtype: int64

In [84]:
from sklearn.model_selection import train_test_split

train_df,test_df = train_test_split(df,test_size=0.2,stratify=df['sentiment'])

In [85]:
vocab = {'<PAD>':0,'<UKN>':1}

def build_vocab(row):
    for token in row['review']:
        if token not in vocab:
            vocab[token] = len(vocab)
            
train_df.apply(build_vocab,axis=1)

22384    None
46888    None
6303     None
27045    None
21691    None
         ... 
27849    None
20444    None
49211    None
17193    None
20858    None
Length: 40000, dtype: object

In [86]:
len(vocab)

193515

In [87]:
def text_to_indices(text,vocab):
    indexed_doc = []
    for token in pre_process_data(text):
        if token in vocab:
            indexed_doc.append(vocab[token])
        else:
            indexed_doc.append(vocab['<UKN>'])
    return indexed_doc

In [88]:
class MyCustomDataset(Dataset):
    
    def __init__(self,df,vocab):
        self.df = df
        self.vocab = vocab
        
    def __len__(self):
        return self.df.shape[0]
        
    def __getitem__(self,index):
        numeric_doc = text_to_indices(self.df.iloc[index]['review'],self.vocab)
        numeric_label = self.df.iloc[index]['sentiment']
        
        return torch.tensor(numeric_doc),torch.tensor(numeric_label)

In [89]:
dataset = MyCustomDataset(df,vocab)

In [ ]:
dataloder = 